In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
df = pd.read_csv("../data/dormmatch_raw.csv")
df_clean = df.copy()

In [22]:
cleaning_log = []

def log_cleaning(step, description, rows_before=None, rows_after=None):
    cleaning_log.append({
        "step": step,
        "description": description,
        "rows_before": rows_before,
        "rows_after": rows_after
    })

In [3]:
rows_before = len(df_clean)
df_clean = df_clean.drop_duplicates()
log_cleaning("Exact duplicates", "Removed exact duplicate rows", rows_before, len(df_clean))
df_clean.shape

(2012, 25)

In [23]:
df_clean.duplicated().sum()

np.int64(0)

In [25]:
df_clean["branch"].value_counts(dropna=False)

branch
Computer Science                 821
Electronics and Communication    271
it                               199
Mechanical Engineering           189
Civil Engineering                148
information technology            82
electrical                        76
biotech                           68
biotechnology                     55
bt                                46
ee                                45
Name: count, dtype: Int64

In [26]:
branch_mapping = {
    "computer science": "Computer Science",
    "electronics and communication": "Electronics and Communication",

    "it": "Information Technology",
    "information technology": "Information Technology",

    "mechanical engineering": "Mechanical Engineering",

    "civil engineering": "Civil Engineering",

    "electrical": "Electrical Engineering",
    "ee": "Electrical Engineering",

    "biotech": "Biotechnology",
    "biotechnology": "Biotechnology",
    "bt": "Biotechnology"
}

In [27]:
df_clean["branch"] = (
    df_clean["branch"]
    .str.strip()
    .str.lower()
    .replace(branch_mapping)
)

In [28]:
df_clean["branch"].value_counts(dropna=False)

branch
Computer Science                 821
Information Technology           281
Electronics and Communication    271
Mechanical Engineering           189
Biotechnology                    169
Civil Engineering                148
Electrical Engineering           121
Name: count, dtype: Int64

In [29]:
df_clean["study_hours"].head(20)

0     2.67
1     2.83
2     5.41
3     3.63
4     4.54
5     5.16
6      3.6
7     1.84
8     0.83
9     4.29
10     4.9
11    3.85
12    3.82
13    4.58
14    2.72
15     3.0
16    2.69
17    1.29
18    2.17
19    4.17
Name: study_hours, dtype: Float64

In [30]:
df_clean["study_hours"] = (
    df_clean["study_hours"]
    .astype(str)
    .str.extract(r"([-+]?\d*\.?\d+)")
)

In [31]:
df_clean["study_hours"] = pd.to_numeric(
    df_clean["study_hours"],
    errors="coerce"
)

In [32]:
df_clean.dtypes

student_id               string[python]
age                               int64
gender                   string[python]
year_of_study                     int64
branch                   string[python]
cgpa                            float64
sleep_time                      float64
wake_time                       float64
sleep_duration                  float64
cleanliness_level               Float64
noise_tolerance                 Float64
study_hours                     float64
social_activity                   Int64
introvert_extrovert             float64
sharing_preference                int64
guest_frequency                 Float64
music_frequency                 Float64
weekend_activity         string[python]
food_habit               string[python]
smoking_preference       string[python]
ac_preference            string[python]
preferred_room_type      string[python]
budget                          float64
preferred_environment    string[python]
roommate_interaction              int64


In [33]:
numeric_columns = [
    "age",
    "cgpa",
    "sleep_time",
    "wake_time",
    "sleep_duration",
    "cleanliness_level",
    "noise_tolerance",
    "study_hours",
    "social_activity",
    "introvert_extrovert",
    "guest_frequency",
    "music_frequency",
    "budget"
]

In [34]:
for col in numeric_columns:
    df_clean[col] = pd.to_numeric(
        df_clean[col],
        errors="coerce"
    )

In [35]:
df_clean[numeric_columns].dtypes

age                      int64
cgpa                   float64
sleep_time             float64
wake_time              float64
sleep_duration         float64
cleanliness_level      Float64
noise_tolerance        Float64
study_hours            float64
social_activity          Int64
introvert_extrovert    float64
guest_frequency        Float64
music_frequency        Float64
budget                 float64
dtype: object

In [36]:
df_clean[
    (df_clean["age"] < 17) |
    (df_clean["age"] > 30)
]

,student_id,age,gender,year_of_study,branch,cgpa,sleep_time,wake_time,sleep_duration,cleanliness_level,...,guest_frequency,music_frequency,weekend_activity,food_habit,smoking_preference,ac_preference,preferred_room_type,budget,preferred_environment,roommate_interaction


In [37]:
df_clean[
    (df_clean["cgpa"] < 0) |
    (df_clean["cgpa"] > 10)
]

,student_id,age,gender,year_of_study,branch,cgpa,sleep_time,wake_time,sleep_duration,cleanliness_level,...,guest_frequency,music_frequency,weekend_activity,food_habit,smoking_preference,ac_preference,preferred_room_type,budget,preferred_environment,roommate_interaction


In [38]:
df_clean[
    (df_clean["study_hours"] < 0) |
    (df_clean["study_hours"] > 24)
]

,student_id,age,gender,year_of_study,branch,cgpa,sleep_time,wake_time,sleep_duration,cleanliness_level,...,guest_frequency,music_frequency,weekend_activity,food_habit,smoking_preference,ac_preference,preferred_room_type,budget,preferred_environment,roommate_interaction


In [39]:
df_clean[
    (df_clean["sleep_duration"] < 0) |
    (df_clean["sleep_duration"] > 24)
]

,student_id,age,gender,year_of_study,branch,cgpa,sleep_time,wake_time,sleep_duration,cleanliness_level,...,guest_frequency,music_frequency,weekend_activity,food_habit,smoking_preference,ac_preference,preferred_room_type,budget,preferred_environment,roommate_interaction


In [40]:
def check_range(df, column, minimum, maximum):
    invalid = df[
        (df[column] < minimum) |
        (df[column] > maximum)
    ]

    print(f"Column: {column}")
    print(f"Invalid values: {len(invalid)}")

    return invalid

In [41]:
check_range(df_clean, "cgpa", 0, 10)

Column: cgpa
Invalid values: 0


,student_id,age,gender,year_of_study,branch,cgpa,sleep_time,wake_time,sleep_duration,cleanliness_level,...,guest_frequency,music_frequency,weekend_activity,food_habit,smoking_preference,ac_preference,preferred_room_type,budget,preferred_environment,roommate_interaction


In [42]:
missing_values = df_clean.isnull().sum()

missing_percentage = (
    df_clean.isnull().sum() / len(df_clean)
) * 100

missing_summary = pd.DataFrame({
    "missing_count": missing_values,
    "missing_percentage": missing_percentage
})

missing_summary.sort_values(
    by="missing_percentage",
    ascending=False
)

,missing_count,missing_percentage
music_frequency,2000,100.0
guest_frequency,2000,100.0
student_id,0,0.0
introvert_extrovert,0,0.0
preferred_environment,0,0.0
budget,0,0.0
preferred_room_type,0,0.0
ac_preference,0,0.0
smoking_preference,0,0.0
food_habit,0,0.0


In [45]:
df_clean[["music_frequency", "guest_frequency"]].isnull().sum()

music_frequency    2000
guest_frequency    2000
dtype: int64

In [64]:
df_clean.drop(
    columns=["music_frequency", "guest_frequency"],
    inplace=True
)

In [65]:
print("Dataset shape:", df_clean.shape)

print("\nRemaining exact duplicates:")
print(df_clean.duplicated().sum())

print("\nDuplicate student IDs:")
print(df_clean["student_id"].duplicated().sum())

print("\nMissing values:")
print(df_clean.isnull().sum())

Dataset shape: (2000, 24)

Remaining exact duplicates:
0

Duplicate student IDs:
0

Missing values:
student_id                   0
age                          0
gender                       0
year_of_study                0
branch                       0
cgpa                         0
sleep_time                   0
wake_time                    0
sleep_duration               0
cleanliness_level            0
noise_tolerance              0
study_hours                  0
social_activity              0
introvert_extrovert          0
sharing_preference           0
weekend_activity             0
food_habit                   0
smoking_preference           0
ac_preference                0
preferred_room_type          0
budget                       0
preferred_environment        0
roommate_interaction         0
calculated_sleep_duration    0
dtype: int64


In [66]:
df_clean.to_csv(
    "../data/dormmatch_cleaned.csv",
    index=False
)